# Smart MCQ Solver — MiniLM Full Fine-Tuning

**Model type :** Full Fine-Tuning of Pretrained Transformer Encoder  
**Base model :** `microsoft/MiniLM-L12-H384-uncased` (~33 M parameters)  
**Task format :** Multiple-Choice QA — score 5 (Prompt, Option) pairs → predict answer label  
**Training    :** End-to-end CrossEntropyLoss optimization with Linear Warmup & Cosine Decay  
**Tracking    :** Weights & Biases (W&B)  
**Metric      :** MAP@3 (Mean Average Precision at 3)

## 1. Install Dependencies

> **Kaggle users:** run this in a separate cell:
> ```
> !pip install -q transformers>=4.40.0 accelerate>=0.28.0 datasets>=2.18.0 wandb sentencepiece scikit-learn
> ```
>
> **Local users:** the cell below uses `uv pip install`.

In [1]:
!pip install -q transformers>=4.40.0 accelerate>=0.28.0 datasets>=2.18.0 wandb sentencepiece scikit-learn

In [2]:
import subprocess, sys, os

PACKAGES = [
    "transformers>=4.40.0",
    "accelerate>=0.28.0",
    "datasets>=2.18.0",
    "wandb",
    "sentencepiece",
    "scikit-learn",
]

if not os.path.exists("/kaggle"):
    print("Installing via uv...")
    subprocess.check_call(["uv", "pip", "install", *PACKAGES])
    print("Dependencies ready ✓")
else:
    print("Kaggle detected — skip this cell and use !pip install instead.")

Kaggle detected — skip this cell and use !pip install instead.


## 2. Imports & Global Configuration

In [3]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
import wandb
from tqdm.auto import tqdm

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ───────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Paths (Kaggle vs Local) ───────────────────────────────────────
if os.path.exists("/kaggle"):
    TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
    TEST_PATH  = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
    OUTPUT_DIR = "/kaggle/working"
else:
    TRAIN_PATH = "../Data/train.csv"
    TEST_PATH  = "../Data/test.csv"
    OUTPUT_DIR = "../Models/minilm_full"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────
OPTIONS       = ["A", "B", "C", "D", "E"]
MODEL_NAME    = "microsoft/MiniLM-L12-H384-uncased"
WANDB_PROJECT = "24F2008474-t22026"

print(f"Device       : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch      : {torch.__version__}")
print(f"Base model   : {MODEL_NAME}")

Device       : cuda
GPU          : Tesla T4
VRAM         : 15.6 GB
PyTorch      : 2.10.0+cu128
Base model   : microsoft/MiniLM-L12-H384-uncased


## 3. W&B Login

In [4]:
wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

## 4. Data Loading & EDA

In [5]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("=" * 55)
print(f"Train : {train_df.shape}   Test : {test_df.shape}")
print("=" * 55)
print(train_df.dtypes)
print()
print("Answer Distribution:")
print(train_df["answer"].value_counts())
print()
sample = train_df.iloc[0]
print(f"  Prompt : {sample['prompt'][:80]}...")
for opt in OPTIONS:
    print(f"  {opt} : {sample[opt][:60]}")
print(f"  Answer : {sample['answer']}")

Train : (2000, 8)   Test : (500, 7)
id         int64
prompt    object
A         object
B         object
C         object
D         object
E         object
answer    object
dtype: object

Answer Distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

  Prompt : Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
  A : Martin Heidegger believes that humans exist within a time co
  B : Martin Heidegger believes that humans do not exist inside ti
  C : Martin Heidegger does not believe in the existence of time o
  D : Martin Heidegger believes that the relationship between time
  E : Martin Heidegger believes that time is an illusion, and the 
  Answer : B


## 5. MAP@3 Metric

In [6]:
def map_at_k(actual: list, predicted: list, k: int = 3) -> float:
    scores = []
    for act, preds in zip(actual, predicted):
        ap = 0.0
        for rank, pred in enumerate(preds[:k], start=1):
            if pred == act:
                ap = 1.0 / rank
                break
        scores.append(ap)
    return float(np.mean(scores))

assert abs(map_at_k(["A"], [["A","B","C"]]) - 1.0) < 1e-9
assert abs(map_at_k(["A"], [["B","A","C"]]) - 0.5) < 1e-9
assert abs(map_at_k(["A"], [["C","D","A"]]) - 1/3) < 1e-9
assert abs(map_at_k(["A"], [["B","C","D"]]) - 0.0) < 1e-9
print("MAP@3 sanity checks passed ✓")

MAP@3 sanity checks passed ✓


## 6. PyTorch MCQ Dataset Class

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME} | Vocab: {tokenizer.vocab_size:,}")

class MCQDatasetMiniLM(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = 256, is_test: bool = False):
        self.df         = df.reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.is_test    = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        input_ids, attention_mask = [], []
        prompt_text = str(row["prompt"])

        for opt in OPTIONS:
            option_text = str(row[opt])
            enc = self.tokenizer(
                prompt_text, option_text,
                max_length=self.max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            input_ids.append(enc["input_ids"].squeeze(0))
            attention_mask.append(enc["attention_mask"].squeeze(0))

        input_ids      = torch.stack(input_ids)       # (5, max_length)
        attention_mask = torch.stack(attention_mask)  # (5, max_length)

        if self.is_test:
            return input_ids, attention_mask

        target = torch.tensor(OPTIONS.index(row["answer"]), dtype=torch.long)
        return input_ids, attention_mask, target

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded: microsoft/MiniLM-L12-H384-uncased | Vocab: 30,522


## 7. Model Architecture (MiniLM MCQ Classifier)

In [8]:
class MiniLMMCQClassifier(nn.Module):
    def __init__(self, model_name: str = MODEL_NAME, dropout_rate: float = 0.2):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(model_name)
        self.dropout    = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, 1)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        # input_ids: (batch_size, 5, seq_length)
        b, n, l = input_ids.size()
        flat_ids   = input_ids.view(b * n, l)
        flat_mask  = attention_mask.view(b * n, l)

        outputs = self.encoder(input_ids=flat_ids, attention_mask=flat_mask)
        cls_rep = outputs.last_hidden_state[:, 0, :]  # [CLS] token representation
        pooled  = self.dropout(cls_rep)
        logits  = self.classifier(pooled).view(b, n)  # (batch_size, 5)
        return logits

model_test = MiniLMMCQClassifier().to(DEVICE)
total_p    = sum(p.numel() for p in model_test.parameters())
print(f"MiniLM Model loaded ✓  |  Total parameters: {total_p / 1e6:.1f} M")
del model_test
torch.cuda.empty_cache()

pytorch_model.bin:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

MiniLM Model loaded ✓  |  Total parameters: 33.4 M


## 8. Dataset Preparation & DataLoaders

In [9]:
train_data, val_data = train_test_split(
    train_df, test_size=0.10, random_state=SEED, stratify=train_df["answer"]
)

BATCH_SIZE  = 8
ACCUM_STEPS = 4
MAX_LEN     = 256

train_ds = MCQDatasetMiniLM(train_data, tokenizer, max_length=MAX_LEN)
val_ds   = MCQDatasetMiniLM(val_data,   tokenizer, max_length=MAX_LEN)
test_ds  = MCQDatasetMiniLM(test_df,    tokenizer, max_length=MAX_LEN, is_test=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train samples : {len(train_ds)}  ({len(train_loader)} batches)")
print(f"Val samples   : {len(val_ds)}    ({len(val_loader)} batches)")
print(f"Test samples  : {len(test_ds)}   ({len(test_loader)} batches)")

Train samples : 1800  (225 batches)
Val samples   : 200    (25 batches)
Test samples  : 500   (63 batches)


## 9. Training Configuration

In [10]:
FINETUNE_CONFIG = {
    "model"                       : MODEL_NAME,
    "finetuning_type"             : "Full Fine-Tuning",
    "num_epochs"                  : 5,
    "batch_size"                  : BATCH_SIZE,
    "gradient_accumulation_steps": ACCUM_STEPS,
    "learning_rate"               : 2e-5,
    "weight_decay"                : 0.01,
    "warmup_ratio"                : 0.1,
    "max_seq_length"              : MAX_LEN,
    "seed"                        : SEED,
}

print("Fine-Tuning Configuration:")
for k, v in FINETUNE_CONFIG.items():
    print(f"  {k:<35} : {v}")

Fine-Tuning Configuration:
  model                               : microsoft/MiniLM-L12-H384-uncased
  finetuning_type                     : Full Fine-Tuning
  num_epochs                          : 5
  batch_size                          : 8
  gradient_accumulation_steps         : 4
  learning_rate                       : 2e-05
  weight_decay                        : 0.01
  warmup_ratio                        : 0.1
  max_seq_length                      : 256
  seed                                : 42


## 10. Training Loop with MAP@3 Validation & W&B Tracking

In [11]:
run = wandb.init(
    project=WANDB_PROJECT,
    name="minilm-full-finetune",
    config=FINETUNE_CONFIG,
    tags=["minilm", "full-finetune", "encoder", "mcq"],
    reinit="finish_previous",
)

model = MiniLMMCQClassifier().to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=FINETUNE_CONFIG["learning_rate"], weight_decay=FINETUNE_CONFIG["weight_decay"])

total_steps  = (len(train_loader) // ACCUM_STEPS) * FINETUNE_CONFIG["num_epochs"]
warmup_steps = int(total_steps * FINETUNE_CONFIG["warmup_ratio"])
scheduler    = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

best_val_map3 = 0.0
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_minilm_model.pt")

print("\nStarting Full Fine-Tuning MiniLM...")
print("=" * 60)

for epoch in range(1, FINETUNE_CONFIG["num_epochs"] + 1):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (input_ids, attention_mask, targets) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{FINETUNE_CONFIG['num_epochs']}")):
        input_ids      = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        targets        = targets.to(DEVICE)

        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, targets) / ACCUM_STEPS
        loss.backward()

        running_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    avg_train_loss = running_loss / len(train_loader)

    # ── Validation Evaluation ──────────────────────────────────────
    model.eval()
    val_logits_list, val_targets_list = [], []
    with torch.no_grad():
        for input_ids, attention_mask, targets in val_loader:
            input_ids      = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            logits         = model(input_ids, attention_mask)
            val_logits_list.append(logits.cpu().numpy())
            val_targets_list.append(targets.numpy())

    val_logits  = np.concatenate(val_logits_list, axis=0)   # (N_val, 5)
    val_targets = np.concatenate(val_targets_list, axis=0)  # (N_val,)

    # Predict top-3 option letters
    val_preds_top3 = [
        [OPTIONS[i] for i in np.argsort(-val_logits[k])[:3]]
        for k in range(len(val_logits))
    ]
    val_actuals = [OPTIONS[t] for t in val_targets]

    val_map3 = map_at_k(val_actuals, val_preds_top3)
    val_acc  = np.mean([p[0] == a for p, a in zip(val_preds_top3, val_actuals)])

    print(f"  Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Val MAP@3: {val_map3:.4f} | Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch,
        "train/loss": avg_train_loss,
        "val/map3": val_map3,
        "val/accuracy": val_acc,
    })

    if val_map3 > best_val_map3:
        best_val_map3 = val_map3
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  --> Best model checkpoint saved! (Val MAP@3: {val_map3:.4f})")

print("=" * 60)
print(f"Fine-Tuning Complete! Best Validation MAP@3: {best_val_map3:.4f}")

wandb: setting up run ga1r9wqo
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260727_170854-ga1r9wqo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run minilm-full-finetune
wandb: ⭐️ View project at https://wandb.ai/spreadsheets600-student/24F2008474-t22026
wandb: 🚀 View run at https://wandb.ai/spreadsheets600-student/24F2008474-t22026/runs/ga1r9wqo


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Starting Full Fine-Tuning MiniLM...


Epoch 1/5:   0%|          | 0/225 [00:00<?, ?it/s]

  Epoch 01 | Train Loss: 1.6003 | Val MAP@3: 0.6650 | Val Acc: 0.5050
  --> Best model checkpoint saved! (Val MAP@3: 0.6650)


Epoch 2/5:   0%|          | 0/225 [00:00<?, ?it/s]

... (3 outputs left out)

Epoch 4/5:   0%|          | 0/225 [00:00<?, ?it/s]

  Epoch 04 | Train Loss: 0.6168 | Val MAP@3: 0.9458 | Val Acc: 0.9250
  --> Best model checkpoint saved! (Val MAP@3: 0.9458)


Epoch 5/5:   0%|          | 0/225 [00:00<?, ?it/s]

  Epoch 05 | Train Loss: 0.5849 | Val MAP@3: 0.9458 | Val Acc: 0.9250
Fine-Tuning Complete! Best Validation MAP@3: 0.9458


## 11. Save Fine-Tuned Model & Tokenizer

In [12]:
# Load best checkpoint weights
model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()

SAVE_DIR = os.path.join(OUTPUT_DIR, "minilm_weights")
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
model.encoder.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved MiniLM fine-tuned encoder and tokenizer → {SAVE_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved MiniLM fine-tuned encoder and tokenizer → /kaggle/working/minilm_weights


## 12. Generate Kaggle Test Predictions

In [13]:
test_logits_list = []
with torch.no_grad():
    for input_ids, attention_mask in tqdm(test_loader, desc="Test Inference"):
        input_ids      = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        logits         = model(input_ids, attention_mask)
        test_logits_list.append(logits.cpu().numpy())

test_logits = np.concatenate(test_logits_list, axis=0)  # (N_test, 5)

test_preds_top3 = [
    " ".join([OPTIONS[i] for i in np.argsort(-test_logits[k])[:3]])
    for k in range(len(test_logits))
]

submission_df = pd.DataFrame({
    "id": test_df["id"].values,
    "Prediction": test_preds_top3,
})

print(submission_df.head(10).to_string(index=False))
print(f"\nSubmission shape : {submission_df.shape}")

Test Inference:   0%|          | 0/63 [00:00<?, ?it/s]

 id Prediction
  1      A E D
  2      B E C
  3      B E D
  4      C E B
  5      C A D
  6      D C B
  7      E C A
  8      B A E
  9      C D E
 10      B E C

Submission shape : (500, 2)


## 13. Save Submission File

In [14]:
SUBMISSION_PATH = "submission_minilm.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"Submission saved → {SUBMISSION_PATH}")
wandb.log({"test/submission_rows": len(submission_df)})
wandb.summary["best_val_map3"] = best_val_map3

Submission saved → submission_minilm.csv


## 14. Error Analysis

In [15]:
val_analysis_df = val_data.copy().reset_index(drop=True)
val_analysis_df["pred_top1"] = [p[0] for p in val_preds_top3]
val_analysis_df["pred_top3"] = [" ".join(p) for p in val_preds_top3]
val_analysis_df["correct"]   = val_analysis_df["pred_top1"] == val_analysis_df["answer"]
val_analysis_df["in_top3"]   = [a in p for a, p in zip(val_analysis_df["answer"], val_preds_top3)]

errors      = val_analysis_df[~val_analysis_df["correct"]]
not_in_top3 = val_analysis_df[~val_analysis_df["in_top3"]]

print(f"Top-1 Errors : {len(errors)}/{len(val_analysis_df)} = {len(errors)/len(val_analysis_df)*100:.1f}%")
print(f"Not Top-3    : {len(not_in_top3)}/{len(val_analysis_df)} = {len(not_in_top3)/len(val_analysis_df)*100:.1f}%")

for _, row in errors.head(5).iterrows():
    print(f"\nQ  : {row['prompt'][:100]}...")
    print(f"True : {row['answer']}   |   Pred Top-3 : {row['pred_top3']}")

Top-1 Errors : 15/200 = 7.5%
Not Top-3    : 5/200 = 2.5%

Q  : What is linear frame dragging?...
True : D   |   Pred Top-3 : C D A

Q  : What was Isaac Newton's explanation for rectilinear propagation of light?...
True : D   |   Pred Top-3 : C B D

Q  : Determine the correct option: What is the purpose of expressing a map's scale as a ratio, such as 1:...
True : E   |   Pred Top-3 : A C D

Q  : Which hand should be used to apply the right-hand rule when tightening or loosening nuts, screws, bo...
True : B   |   Pred Top-3 : D A B

Q  : Which of the following is correct? What is the term used to describe the proportionality factor to t...
True : A   |   Pred Top-3 : B D A


## 15. Finish W&B Run

In [16]:
wandb.finish()
print("W&B run finished ✓")

wandb: updating run metadata
wandb: uploading history steps 5-5, summary, console lines 15-45
wandb: 
wandb: Run history:
wandb:                epoch ▁▃▅▆█
wandb: test/submission_rows ▁
wandb:           train/loss █▅▂▁▁
wandb:         val/accuracy ▁▆███
wandb:             val/map3 ▁▆███
wandb: 
wandb: Run summary:
wandb:        best_val_map3 0.94583
wandb:                epoch 5
wandb: test/submission_rows 500
wandb:           train/loss 0.58489
wandb:         val/accuracy 0.925
wandb:             val/map3 0.94583
wandb: 
wandb: 🚀 View run minilm-full-finetune at: https://wandb.ai/spreadsheets600-student/24F2008474-t22026/runs/ga1r9wqo
wandb: ⭐️ View project at: https://wandb.ai/spreadsheets600-student/24F2008474-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260727_170854-ga1r9wqo/logs


W&B run finished ✓
